- 2D 坐标: uv，图像上的像素坐标 $(u, v)$。
- 相机参数 (CameraInfo):
    - 内参 (Intrinsics): fx, fy (焦距), cx, cy (光心)。
    - 外参 (Extrinsics):
        - position: 相机在世界坐标系的位置。
        - quaternion: 相机的旋转四元数。

### 位姿

- 在 3D 空间中描述物体的旋转和姿态，最符合人类直觉的是欧拉角 (Euler Angles)（即 Roll, Pitch, Yaw）。然而欧拉角存在万向节死锁 (Gimbal Lock) 的致命缺陷。因此，现代计算机图形学和机器人技术（如 ROS）通常采用四元数 (Quaternion) $q = w + x\mathbf{i} + y\mathbf{j} + z\mathbf{k}$ 来存储姿态。
- 四元素转旋转矩阵
    - $R = \begin{bmatrix}
1 - 2y^2 - 2z^2 & 2xy - 2wz & 2xz + 2wy \\
2xy + 2wz & 1 - 2x^2 - 2z^2 & 2yz - 2wx \\
2xz - 2wy & 2yz + 2wx & 1 - 2x^2 - 2y^2
\end{bmatrix}$
- 要获取实体的“正前方面向哪”，最直接的方法是将四元数转换为 $3 \times 3$ 的旋转矩阵 $R$，然后用它去旋转一个基准的“正前方向量” 

### 3d -> 2d (project)

### 2d -> 3d (back-project)

> 假如投影到一个3d的表面，从相机眼睛发射一道激光，击中物体表面。

- 从像素坐标到相机坐标系射线 (Pixel -> Camera Ray):
    - 利用内参将像素点 $(u, v)$ 转换为相机坐标系下的**方向向量**（如果有 depth 时，则直接计算 $z_c=\text{depth}$，真是物理点）：
    $$x_c = (u - c_x) / f_x$$
    $$y_c = (v - c_y) / f_y$$
    $$z_c = 1.0$$
    - 此时得到相机坐标系下的射线方向 $D_{cam} = (x_c, y_c, z_c)$。
- 从相机坐标系到世界坐标系 (Camera -> World):
    - 利用相机外参（旋转四元数）将射线方向旋转到世界坐标系。
        - Unreal: X=前, Y=右, Z=上
        - Vision: X=右, Y=下, Z=前
    - `ray_dir_world = rotation.apply(convert_coords(ray_dir_cam))`
- 射线与平面求交 (Ray-Plane Intersection):
    - 计算从相机位置 $O$ 出发，沿方向 $D_{world}$ 射出的射线与平面 $Z = z_{world}$ 的交点。
    - 由直线方程 $P = O + t \cdot D$ 和平面方程 $P_z = z_{world}$ 解出 $t$：
    $$t = \frac{z_{world} - O_z}{D_z}$$
    - 最终 3D 坐标为：$P_{intersection} = O + t \cdot D_{world}$。